Import all needed libraries

In [1]:
import networkx as nx
import random
import math
import numpy as np
import pandas as pd
import markov_clustering as mc
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from cdlib.benchmark import LFR
from node2vec import Node2Vec
from cdlib import algorithms, evaluation, NodeClustering

C:\Users\ricca\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Note: to be able to use all crisp methods, you need to install some additional packages:  {'graph_tool', 'bayanpy', 'wurlitzer', 'infomap'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'ASLPAw', 'pyclustering'}
Note: to be able to use all crisp methods, you need to install some additional packages:  {'wurlitzer', 'infomap'}


Define graph generation function:

INPUT:  size 'N' of the graph; mixing parameter 'mu'

OUTPUT: networkx graph; set of ground truth communities

The function generates a LFR graph with the input parameters

In [2]:
def generate_original_graph(N, mu):
    attempt = 1
    while True:
        try:  # needed because sometimes LFR is not able to assign communities
            # generate LFR graph
            print(f"      [LFR] Generation attempt {attempt}...", end="\r")

            avg_deg = random.randint(math.floor(N / 50), math.ceil(N / 25))

            G, ground_truth_comms = LFR(n = N, mu = mu,
                                        tau1 = 3, tau2 = 2,
                                        min_community = int(avg_deg + 10),
                                        average_degree = avg_deg,
                                        max_degree = math.ceil(N/10),
                                        seed = random.randint(1, 10000)
                                        )
            # if successful remove self loops and return the graph + communities
            G.remove_edges_from(nx.selfloop_edges(G))
            print(f"      [LFR] Success on attempt {attempt}              ")
            return G, ground_truth_comms

        # if errors occur retry / give warning
        except nx.ExceededMaxIterations:
            print(f"      [LFR] Attempt {attempt} failed (Max Iterations). Retrying...")
            attempt += 1
        except Exception as e:
            print(f"      [LFR] Unexpected error: {e}")
            raise e

Define graph reconstruction function:

INPUT: networkx graph 'G', numpy array 'X'; reconstruction method 'reconstruction' ("threshold" or "knn"); evaluation metric 'metric' ("dot" or "cosine")

OUTPUT: networkx graph

The function, based on reconstruction and metric, reconstructs the graph from the embeddings saved in X

In [3]:
def get_reconstructed_graph(G, X, reconstruction, metric):

    # if input is not correct stop the execution
    if (reconstruction != "knn" and reconstruction != "threshold") or (metric != "cosine" and metric != "dot"):
        print("Incorrect reconstruction method or metric")
        return None

    # initialize empty graph and edge list
    nodes = sorted(list(G.nodes())) # get nodes list from original graph
    reconstructed_graph = nx.Graph()
    reconstructed_graph.add_nodes_from(nodes)
    edge_list = []

    # based on the selected reconstruction method and metric build the graph
    if reconstruction == "knn":
        # get k for k-NN as the average node degree
        k = int(np.round((2 * G.number_of_edges()) / G.number_of_nodes()))

        if metric == "cosine":
            # get knn model
            knn = NearestNeighbors(n_neighbors=k + 1, # use k+1 because the first neighbor is always the node itself (distance=0)
                                   metric=metric)
            knn.fit(X)
            distances, indices = knn.kneighbors(X)

            # get the list of edges and weights to add to the reconstructed graph
            for i, (neighbor_indices, neighbor_dists) in enumerate(zip(indices, distances)):
                u = nodes[i]
                for neighbor_idx, dist in zip(neighbor_indices[1:], neighbor_dists[1:]):
                    v = nodes[neighbor_idx]
                    weight = 1 - dist # convert distance to similarity
                    edge_list.append((u, v, {'weight': weight}))

        elif metric == "dot":
            # compute similarity matrix and fill diagonal with -inf to avoid self loops
            sim_matrix = np.dot(X, X.T)
            np.fill_diagonal(sim_matrix, -np.inf)

            # for every node find the k highest values
            # argpartition puts the top k indices at the end of the array
            # take the last k indices
            top_k_indices = np.argpartition(sim_matrix, -k, axis=1)[:, -k:]

            # get the list of edges and weights to add to the reconstructed graph
            for i, neighbor_indices in enumerate(top_k_indices):
                u = nodes[i]
                for neighbor_idx in neighbor_indices:
                    v = nodes[neighbor_idx]
                    weight = sim_matrix[i, neighbor_idx] # get the dot product as weight
                    edge_list.append((u, v, {'weight': float(weight)}))

    else: # if reconstruction == "thresholding"
        t = 0.5  # set threshold
        if metric == "cosine":
            # compute cosine similarity matrix and remove self loops
            sim_matrix = cosine_similarity(X)
            np.fill_diagonal(sim_matrix, 0)

            # gets indices of node couples that overcome the threshold that are
            # located in the upper triangle of the matrix (avoids duplicates)
            rows, cols = np.where(np.triu(sim_matrix, k=1) >= t)

            # get the list of edges and weights to add to the reconstructed graph
            for u_idx, v_idx in zip(rows, cols):
                u, v = nodes[u_idx], nodes[v_idx]
                weight = sim_matrix[u_idx, v_idx]
                edge_list.append((u, v, {'weight': float(weight)}))

        elif metric == "dot":
            # compute similarity matrix and fill diagonal with -inf to avoid self loops
            sim_matrix = np.dot(X, X.T)
            np.fill_diagonal(sim_matrix, -np.inf)

            # do min-max normalization to bring range to [0, 1]
            valid_values = sim_matrix[sim_matrix != -np.inf] # ignore -inf values
            min_val = valid_values.min()
            max_val = valid_values.max()
            sim_matrix_norm = (sim_matrix - min_val) / (max_val - min_val)

            # apply threshold (same line of code of cosine)
            rows, cols = np.where(np.triu(sim_matrix_norm, k=1) >= t)

            # get the list of edges and weights to add to the reconstructed graph
            for u_idx, v_idx in zip(rows, cols):
                u, v = nodes[u_idx], nodes[v_idx]
                weight = sim_matrix_norm[u_idx, v_idx]
                edge_list.append((u, v, {'weight': float(weight)}))

    # add edges to the graph
    reconstructed_graph.add_edges_from(edge_list)

    return reconstructed_graph

Define MCL function

INPUT: networkx graph 'G'

OUTPUT: communities list of lists

Runs MCL with modularity parameter optimization

In [4]:
def get_mcl_communities(G):

    matrix = nx.to_numpy_array(G) # get network in matrix form

    # initialize values to do the modularity optimization
    best_modularity = -1
    best_clusters = None
    inflations = np.round(np.arange(1.5, 2.6, 0.1), 2)

    print(f"      [MCL] Optimizing over {len(inflations)} inflation values...", end=" ")

    # for each inflation value test the output of MCL
    for inflation in inflations:
        print(f".", end="", flush=True)

        # run MCL and get the results
        result = mc.run_mcl(matrix, inflation=inflation)
        clusters = mc.get_clusters(result)
        communities = [list(c) for c in clusters]

        # if MCL obtained 0 or 1 communities retry (degenerate/trivial solutions)
        if len(communities) <= 1:
            continue

        try: # block added to avoid networkx errors
            # compute the moduarity based on the MCL output
            Q = nx.algorithms.community.modularity(G, communities)
        except Exception:
            Q = -1

        # update the best values
        if Q > best_modularity:
            best_modularity = Q
            best_clusters = communities

    # if no result is optimal return a default result
    if best_clusters is None:
        result = mc.run_mcl(matrix, inflation=2.0)
        best_clusters = [list(c) for c in mc.get_clusters(result)]

    print(" Done.")
    return best_clusters

Define the function for graph evaluation:



INPUT: all specifics of the experiment, predicted communities 'pred', ground truth communities 'ground_truth'



OUTPUT: a dictionary containing the specifics of the experiment and nmi, ari, f1 score



The function computes the three evaluation metrics for the input experiment

In [5]:
def evaluate_algorithm(algo_name, pred_raw, ground_truth, graph_name, N, mu, method_label, test_id, G_ref):

    # when the input is a result of MCL, we need to transform it into a NodeClustering object
    if isinstance(pred_raw, list):
        pred_obj = NodeClustering(pred_raw, graph=G_ref)
    else:
        pred_obj = pred_raw

    # compute the three evaluation metrics
    try: # if any error happens return 0,0,0 (avoids crashes)
        nmi = evaluation.normalized_mutual_information(pred_obj, ground_truth).score
        ari = evaluation.adjusted_rand_index(pred_obj, ground_truth).score
        f1  = evaluation.f1(pred_obj, ground_truth).score
    except Exception as e:
        print(f"   [Error] {algo_name} evaluation failed: {e}")
        nmi, ari, f1 = 0, 0, 0

    return {
        "graph": graph_name,
        "N": N,
        "mu": mu,
        "method": method_label,
        "algorithm": algo_name,
        "test": test_id,
        "NMI": nmi,
        "F_score": f1,
        "ARI": ari,
    }

Define the set of experiments

In [6]:
N_TESTS = 20 # number of tests for each (graph, method) combination

# define the graphs the experiments will be run on
GRAPHS = {
    "Graph_1": {"N": 500, "mu": 0.1},
    "Graph_2": {"N": 1000, "mu": 0.1},
    "Graph_3": {"N": 1000, "mu": 0.3},
}

# define the methods the experiments will be run on
METHODS = [
    ("knn", "dot"),
    ("knn", "cosine"),
    ("threshold", "dot"),
    ("threshold", "cosine"),
]

Execute the experiments

In [7]:
all_results = [] # initialize empty list for the results

print(f"Starting Experiments: {len(GRAPHS)} Graph types, {N_TESTS} tests each.\n")

# for every graph type...
for graph_name, params in GRAPHS.items():
    print(f"--- Processing {graph_name} (N={params['N']}, mu={params['mu']}) ---")
    current_graph_rows = [] # initialize a list for the results of the actual graph type

    # for every test in [1, N_TESTS]...
    for test_id in range(1, N_TESTS + 1):
        print(f"   > Running Test {test_id}/{N_TESTS}...")

        # generate LFR graph
        G, ground_truth = generate_original_graph(
            N=params["N"],
            mu=params["mu"]
        )

        # run Leiden on the original graph and append results
        leiden_base = algorithms.leiden(G)
        all_results.append(evaluate_algorithm("Leiden", leiden_base, ground_truth, graph_name, params["N"], params["mu"], "Original", test_id, G))
        current_graph_rows.append(all_results[-1])

        # run MCL on the original graph and append results
        mcl_base = get_mcl_communities(G)
        all_results.append(evaluate_algorithm("MCL", mcl_base, ground_truth, graph_name, params["N"], params["mu"], "Original", test_id, G))
        current_graph_rows.append(all_results[-1])

        # compute node2vec embeddings
        node2vec_model = Node2Vec(G, dimensions=64,
                                  q=0.5, p=1)
        embeddings = node2vec_model.fit() # output works like a dictionary, key = node_id

        # convert embeddings into a numpy array
        model_wv = embeddings.wv  # this gets all keyed vectors in the attribute .wv
        nodes = sorted(list(G.nodes()))  # get nodes list from original graph
        X = np.array([model_wv[str(n)] for n in nodes])  # do the lookup in model_wv node by node,
                                                         # convert 'n' to str() because node2vec stores keys as strings

        # for every reconstruction method...
        for reconstruction, metric in METHODS:
            method_label = f"{reconstruction}+{metric}" # get the method type

            # obtain reconstructed graph
            Gr = get_reconstructed_graph(G, X, reconstruction=reconstruction, metric=metric)

            # run Leiden on the reconstructed graph and append results
            leiden_res = algorithms.leiden(Gr)
            all_results.append(evaluate_algorithm("Leiden", leiden_res, ground_truth, graph_name, params["N"], params["mu"], method_label, test_id, G))
            current_graph_rows.append(all_results[-1])

            # run MCL on the reconstructed graph and append results
            mcl_res = get_mcl_communities(Gr)
            all_results.append(evaluate_algorithm("MCL", mcl_res, ground_truth, graph_name, params["N"], params["mu"], method_label, test_id, G))
            current_graph_rows.append(all_results[-1])

    # save partial results (per graph type)
    df_graph = pd.DataFrame(current_graph_rows)

    # create a dataframe to save the mean results of the experiments
    mean_df_graph = ( # calculate mean grouping results by method and algorithm
        df_graph.groupby(["method", "algorithm"])[["NMI", "F_score", "ARI"]]
        .mean()
        .reset_index()
    )
    mean_df_graph["graph"] = graph_name
    mean_df_graph["N"] = params["N"]
    mean_df_graph["mu"] = params["mu"]
    mean_df_graph["test"] = "MEAN"

    # save the partial results in a .csv file
    df_out_graph = pd.concat([df_graph, mean_df_graph], ignore_index=True)
    out_path = f"results_{graph_name.lower()}.csv"
    df_out_graph.to_csv(out_path, index=False)
    print(f"   [Saved checkpoint: {out_path}]\n")

# create a dataframe with all the results and save it in a .csv file
df_all = pd.DataFrame(all_results)
df_all.to_csv("experiment_FULL_RESULTS.csv", index=False)
print("Experiments completed. Full dataset saved.")

Starting Experiments: 3 Graph types, 20 tests each.

--- Processing Graph_1 (N=500, mu=0.1) ---
   > Running Test 1/20...
      [LFR] Success on attempt 1              
      [MCL] Optimizing over 11 inflation values... ........... Done.


Generating walks (CPU: 1): 100%|██████████| 10/10 [00:02<00:00,  4.60it/s]


      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
   > Running Test 2/20...
      [LFR] Success on attempt 1              
      [MCL] Optimizing over 11 inflation values... ........... Done.


Generating walks (CPU: 1): 100%|██████████| 10/10 [00:01<00:00,  5.44it/s]


      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
   > Running Test 3/20...
      [LFR] Success on attempt 1              
      [MCL] Optimizing over 11 inflation values... ........... Done.


...

      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
   > Running Test 19/20...
      [LFR] Success on attempt 1              
      [MCL] Optimizing over 11 inflation values... ........... Done.


Generating walks (CPU: 1): 100%|██████████| 10/10 [00:10<00:00,  1.04s/it]


      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
   > Running Test 20/20...
      [LFR] Success on attempt 1              
      [MCL] Optimizing over 11 inflation values... ........... Done.


Generating walks (CPU: 1): 100%|██████████| 10/10 [00:10<00:00,  1.03s/it]


      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
      [MCL] Optimizing over 11 inflation values... ........... Done.
   [Saved checkpoint: results_graph_3.csv]

Experiments completed. Full dataset saved.


Print summary of the results

In [8]:
# header
print("\n" + "="*65)
print(f"{'FINAL RESULTS SUMMARY (Mean over {N_TESTS} runs)':^65}")
print("="*65)

# calculate the mean of metrics for every group
summary = df_all.groupby(['algorithm', 'method', 'graph'])[['NMI', 'F_score', 'ARI']].mean().reset_index()

algo_order = ["Leiden", "MCL"] # define order for printing

# sort methods to have 'Original' first
method_list = sorted(summary['method'].unique())
if "Original" in method_list:
    method_list.remove("Original")
    method_list.insert(0, "Original")

# print all the results
for algo in algo_order:
    print(f"\n>> ALGORITHM: {algo.upper()}")
    print("-" * 65)

    for meth in method_list:
        # get data for this specific algo + method combination
        subset = summary[(summary['algorithm'] == algo) & (summary['method'] == meth)]

        if subset.empty:
            continue
        print(f"   [Method: {meth}]")
        subset = subset.sort_values("graph") # sort by graph name

        for _, row in subset.iterrows():
            g_name = row['graph']
            nmi = row['NMI']
            f1  = row['F_score']
            ari = row['ARI']

            # print formatted row
            print(f"      {g_name:<10} | NMI: {nmi:.4f} | F1: {f1:.4f} | ARI: {ari:.4f}")
        print("") # empty line for spacing


        FINAL RESULTS SUMMARY (Mean over {N_TESTS} runs)         

>> ALGORITHM: LEIDEN
-----------------------------------------------------------------
   [Method: Original]
      Graph_1    | NMI: 0.9998 | F1: 0.9999 | ARI: 0.9998
      Graph_2    | NMI: 1.0000 | F1: 1.0000 | ARI: 1.0000
      Graph_3    | NMI: 0.9985 | F1: 0.9987 | ARI: 0.9974

   [Method: knn+cosine]
      Graph_1    | NMI: 0.9994 | F1: 0.9996 | ARI: 0.9992
      Graph_2    | NMI: 1.0000 | F1: 1.0000 | ARI: 1.0000
      Graph_3    | NMI: 0.9972 | F1: 0.9982 | ARI: 0.9966

   [Method: knn+dot]
      Graph_1    | NMI: 0.9995 | F1: 0.9997 | ARI: 0.9994
      Graph_2    | NMI: 0.9999 | F1: 0.9999 | ARI: 0.9999
      Graph_3    | NMI: 0.9980 | F1: 0.9988 | ARI: 0.9974

   [Method: threshold+cosine]
      Graph_1    | NMI: 0.9990 | F1: 0.9989 | ARI: 0.9975
      Graph_2    | NMI: 1.0000 | F1: 1.0000 | ARI: 1.0000
      Graph_3    | NMI: 0.9419 | F1: 0.4374 | ARI: 0.9235

   [Method: threshold+dot]
      Graph_1    | NM